# SemiTNet — Reproduction and Evaluation
این Notebook گزارش پژوهشگرمحور سه مسیر مجزاست: **reduced measured simulation**، **strict paper-equivalence (G0–G3)** و **TED3 defensible reimplementation**.

`Images → Encoder → Segmentation/Queries → Teacher–Student → Pseudo Labels → EMA/Distillation → IoU/Dice/Precision/Recall/F1`

> full 26,250-iteration training پیش‌فرض خاموش است؛ هیچ dataset/package دانلود یا extract نمی‌شود.

## 1. Title and Project Overview
SemiTNet مسئله‌ی semi-supervised dental segmentation/identification را هدف می‌گیرد. reduced `QuickSemiTransformer` برای evidence اجرایی است و با publication architecture یکسان نیست.

## 2. Environment Validation
فقط Python/PyTorch/CUDA/GPU/dependency/root بررسی می‌شود؛ install انجام نمی‌شود.

In [ ]:
# هدف: اعتبارسنجی environment و تعیین root.
# ورودی: محیط فعلی.
# خروجی: versions/status/path.
# رفتار مورد انتظار: read-only.
from pathlib import Path
import os,sys,json,subprocess,hashlib,importlib.util,platform,shlex
ROOT=next((p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'project.py').is_file() and (p/'reproduction/reference_contract.json').is_file()),None)
if ROOT is None: raise RuntimeError('repository root not found')
os.chdir(ROOT); RUN_SAFE=False; RUN_REDUCED=False; RUN_HASH=False; RUN_FULL_26250=False
print('ROOT',ROOT); print('Python',sys.version.replace('\n',' ')); print('Platform',platform.platform())
for n in ['torch','numpy','pandas','matplotlib','PIL','yaml','detectron2','pycocotools']: print(n,importlib.util.find_spec(n) is not None)
try:
 import torch; print('PyTorch',torch.__version__,'CUDA',torch.cuda.is_available(),'GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
except Exception as e: print(e)
print('HEAD',subprocess.run(['git','rev-parse','HEAD'],text=True,capture_output=True).stdout.strip())


## 3. Repository Structure Explorer
`scripts/` اجرا؛ `configs/` تنظیمات؛ `data/` ورودی؛ `outputs/` evidence؛ `reproduction/` قرارداد علمی؛ `notebooks/` گزارش.

In [ ]:
# هدف: نمایش tree محدود repository.
# ورودی: ROOT.
# خروجی: مسیرهای مهم.
# رفتار مورد انتظار: dataset بزرگ truncate می‌شود.
for n in ['scripts','configs','data','outputs','reproduction','notebooks']:
 b=ROOT/n; print('\n#',n,b.exists())
 if b.exists():
  xs=[p for p in sorted(b.rglob('*')) if len(p.relative_to(b).parts)<=2][:70]
  for p in xs: print('  '*(len(p.relative_to(b).parts)-1)+('DIR ' if p.is_dir() else 'FILE ')+p.name)


## 4. Dataset Documentation
Authoritative local paths: `data/train/` (labeled)، `data/test/` (held-out)، `data/TED3-unlabeled-data-15k-pseudo-mask/` (unlabeled/pseudo-mask). داده دوباره extract/download نمی‌شود و pseudo-mask بدون provenance، human ground truth نیست.

In [ ]:
# هدف: inventory، manifest، نمونه image/annotation و leakage audit اختیاری.
# ورودی: سه TED3 directory.
# خروجی: counts/sizes/samples/manifests/duplicate status.
# رفتار مورد انتظار: read-only؛ SHA کامل opt-in.
from collections import Counter
import pandas as pd,numpy as np,matplotlib.pyplot as plt
from PIL import Image
D={'train':ROOT/'data/train','test':ROOT/'data/test','unlabeled':ROOT/'data/TED3-unlabeled-data-15k-pseudo-mask'}; EXT={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
rows=[]
for r,p in D.items():
 f=[x for x in p.rglob('*') if x.is_file()] if p.is_dir() else []
 rows.append([r,str(p.relative_to(ROOT)),p.is_dir(),len(f),round(sum(x.stat().st_size for x in f)/1024**3,3),dict(Counter(x.suffix.lower() for x in f).most_common(5))])
display(pd.DataFrame(rows,columns=['role','path','exists','files','GiB','extensions']))
manifests=[p for b in [ROOT/'data/processed/ted3',ROOT/'outputs/ted3_reproduction'] if b.exists() for p in b.rglob('*manifest*.json')]
print('manifests:',*[str(p.relative_to(ROOT)) for p in manifests[:20]],sep='\n- ')
for r,p in D.items():
 im=next((x for x in sorted(p.rglob('*')) if x.is_file() and x.suffix.lower() in EXT),None) if p.is_dir() else None
 if im: plt.figure(figsize=(8,3)); plt.imshow(Image.open(im),cmap='gray'); plt.title(str(im.relative_to(ROOT))); plt.axis('off'); plt.show()
 an=next(iter(p.rglob('*.json')),None) if p.is_dir() else None
 if an:
  try:o=json.loads(an.read_text()); print(r,'annotation example',an.relative_to(ROOT),'keys',list(o)[:12] if isinstance(o,dict) else type(o))
  except: pass
if RUN_HASH:
 def H(x):
  h=hashlib.sha256()
  with x.open('rb') as f:
   for z in iter(lambda:f.read(8*1024*1024),b''): h.update(z)
  return h.hexdigest()
 hs={r:{H(x) for x in p.rglob('*') if x.is_file() and x.suffix.lower() in EXT} for r,p in D.items()}
 for a,b in [('train','test'),('train','unlabeled'),('test','unlabeled')]: print(a,b,len(hs[a]&hs[b]))


## 5. Paper-to-Code Mapping
Published contract: **32 classes، RGB 1024، 100 queries، 9 decoder layers، AdamW، LR=1e-4، steps 24000/25000، 26,250 iterations**. `burnin_iter=3000` در مخزن inferred است.

In [ ]:
# هدف: mapping واقعی paper component به code.
# ورودی: reference_contract/config/source.
# خروجی: config و mapping tables.
# رفتار مورد انتظار: reduced ≠ paper architecture.
contract=json.loads((ROOT/'reproduction/reference_contract.json').read_text())
try:
 import yaml; cfg=yaml.safe_load((ROOT/'configs/paper.yaml').read_text())
except: cfg={}
display(pd.DataFrame([{'parameter':k,'value':v} for k,v in cfg.items()]))
M=[['Backbone/Encoder','QuickSemiTransformer patch+TransformerEncoder','run_quick_real_experiment.py','reduced; paper MobileSAM/TinyViT'],['Decoder/Queries','ConvTranspose reduced; paper 100 queries/9 layers','quick script + paper.yaml','not equivalent'],['Heads','33 channels: bg+32 teeth','QuickSemiTransformer','reduced'],['Losses','weighted CE + 0.5 Dice','supervised_loss','reduced'],['Teacher/Student','warm-up→copy→pseudo supervision','quick script::main','reduced'],['Pseudo labels','confidence; low=-100 ignore','quick script::main','no forced background'],['EMA/Distillation','.995T+.005S; sup+.05unsup','quick script::main','paper cfg differs'],['Full training','vendor teacher→student','scripts/run_full.py','expensive opt-in'],['Checkpoint eval','vendor eval→COCO→metrics','scripts/run_official_inference.py','compatible data/ckpt required'],['Metrics','COCOeval@.5; Dice segm; P/R/F1 bbox','compute_article_metrics.py','paper-style'],['Optimizer/Scheduler','AdamW + LR/steps','configs/paper.yaml','publication target']]
display(pd.DataFrame(M,columns=['Paper Component','Implementation','Location','Note']))


## 6. Simulation Pipeline Explanation
Reduced: `PreparedTeeth → Teacher → CE+Dice → pseudo filtering → Student sup+unsup → EMA → collapse guard → evaluation → JSON/CSV/PNG`.

Full: `scripts/run_full.py` teacher سپس student را روی pinned vendor اجرا می‌کند.

TED3 contract: `preflight → forensic audit/prep → method freeze → checkpoint eval → supervised → semi-supervised → comparison → exports → delivery`.

In [ ]:
# هدف: safe runner و stage matrix کل simulation.
# ورودی: flags، scripts و artifacts.
# خروجی: command/status هر مرحله.
# رفتار مورد انتظار: full training فقط opt-in.
CMDS={'preflight':[sys.executable,'project.py','ted3-preflight'],'audit':[sys.executable,'project.py','audit'],'reduced':[sys.executable,'scripts/run_quick_real_experiment.py'],'figures':[sys.executable,'scripts/export_paper_style_figures.py'],'full':[sys.executable,'scripts/run_full.py','--dataset','data/processed/ted3']}
def run(n):
 if n in {'preflight','audit','figures'} and not RUN_SAFE:return print('[SKIP]',n)
 if n=='reduced' and not RUN_REDUCED:return print('[SKIP] reduced')
 if n=='full' and not RUN_FULL_26250:return print('[SKIP] full 26,250')
 p=subprocess.run(CMDS[n],text=True,capture_output=True); print(' '.join(map(shlex.quote,CMDS[n])),p.returncode); print(p.stdout[-2000:],p.stderr[-1000:])
if RUN_SAFE: run('preflight'); run('audit')
if RUN_REDUCED: run('reduced')
if RUN_FULL_26250: run('full')
ph=[('0 preflight',['scripts/ted3_input_preflight.py'],['outputs/ted3_reproduction/preflight']),('1 audit/prep',[],['outputs/ted3_reproduction/dataset_audit','data/processed/ted3']),('2 freeze',[],['reproduction/paper_method_manifest.json']),('3 checkpoint',['scripts/run_official_inference.py'],['outputs/ted3_reproduction/checkpoint_eval','outputs/inference']),('4–5 teacher/student',['scripts/run_full.py'],['outputs/ted3_reproduction/supervised','outputs/ted3_reproduction/semi_supervised','outputs/full']),('6 comparison',['scripts/compare_to_paper.py'],['outputs/ted3_reproduction/comparison']),('7 exports',['scripts/export_paper_style_figures.py'],['outputs/ted3_reproduction/paper_exports','outputs/paper_style']),('8 delivery',['scripts/package_client_delivery.py'],['outputs/ted3_reproduction/delivery'])]
display(pd.DataFrame([[n,[x for x in s if (ROOT/x).is_file()],[x for x in o if (ROOT/x).exists()]] for n,s,o in ph],columns=['phase','entrypoints','artifacts']))


## 7. Load Existing Experimental Results
Measured metrics و manifests از outputهای موجود خوانده می‌شوند؛ missing result ساخته نمی‌شود.

In [ ]:
# هدف: catalog metrics/manifests.
# ورودی: outputs موجود.
# خروجی: tables.
# رفتار مورد انتظار: فقط JSON واقعی.
MET=['iou','dice','precision','recall','f1']; rec=[]; mans=[]
for b in [ROOT/'outputs/ted3_reproduction',ROOT/'outputs/paper_reproduction',ROOT/'outputs/inference',ROOT/'outputs/final']:
 if b.exists():
  for p in b.rglob('*.json'):
   try:o=json.loads(p.read_text())
   except:continue
   if isinstance(o,dict) and all(isinstance(o.get(k),(int,float)) for k in MET): rec.append({'source':str(p.relative_to(ROOT)),**{k:float(o[k]) for k in MET}})
   if 'manifest' in p.name.lower(): mans.append([str(p.relative_to(ROOT)),list(o)[:12] if isinstance(o,dict) else []])
display(pd.DataFrame(rec)); display(pd.DataFrame(mans,columns=['manifest','keys']))


## 8. Paper Comparison
Published SemiTNet و measured artifact با Gap و source جدا نمایش داده می‌شوند.

In [ ]:
# هدف: Published/Measured/Gap.
# ورودی: contract و rec.
# خروجی: comparison table.
# رفتار مورد انتظار: missing measured خالی.
prio=['semi_supervised','supervised','checkpoint_eval','outputs/inference','outputs/final']; sel=next((r for t in prio for r in rec if t in r['source']),None); pub=contract['publication']['published_metrics_percent']
display(pd.DataFrame([[k.upper(),pub[k],None if not sel else sel[k],None if not sel else sel[k]-pub[k],None if not sel else sel['source']] for k in MET],columns=['Metric','Published SemiTNet','Measured Result','Gap','Source']))


## 9. Visualization Gallery
Training/loss curves و qualitative/segmentation figures موجود خودکار نمایش داده می‌شوند.

In [ ]:
# هدف: نمایش curves و gallery.
# ورودی: history.csv و figures.
# خروجی: matplotlib figures.
# رفتار مورد انتظار: فقط artifact موجود.
for hp in (ROOT/'outputs').rglob('history.csv') if (ROOT/'outputs').exists() else []:
 try:h=pd.read_csv(hp)
 except:continue
 if 'loss' in h: plt.figure(figsize=(8,3)); plt.plot(h['loss']); plt.title(str(hp.relative_to(ROOT))); plt.ylabel('Loss'); plt.grid(alpha=.2); plt.show()
dirs=[ROOT/'outputs/ted3_reproduction/compact_supervised/figures',ROOT/'outputs/ted3_reproduction/compact_semi_supervised/figures',ROOT/'outputs/ted3_reproduction/supervised/figures',ROOT/'outputs/ted3_reproduction/semi_supervised/figures',ROOT/'outputs/final/figures',ROOT/'outputs/inference/figures',ROOT/'outputs/paper_style/figures']
figs=[p for d in dirs if d.is_dir() for p in d.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'}]
for p in figs[:20]: plt.figure(figsize=(9,4)); plt.imshow(Image.open(p)); plt.title(str(p.relative_to(ROOT))); plt.axis('off'); plt.show()


## 10. Reproducibility Section
Safe:
`python project.py ted3-preflight` · `python project.py audit` · `python scripts/export_paper_style_figures.py`

Reduced:
`python scripts/run_quick_real_experiment.py` سپس `python scripts/validate_final_outputs.py`

Full expensive:
`python scripts/run_full.py --dataset <compatible_processed_root> --num-gpus <N> --batch-size <B>`

Official checkpoint:
`python scripts/run_official_inference.py --dataset <compatible_processed_root> --checkpoint <checkpoint.pth> --num-gpus 1`


In [ ]:
# هدف: G0–G3، seeds و checklist.
# ورودی: audit/config/manifests/artifacts.
# خروجی: reproducibility status.
# رفتار مورد انتظار: G0 PASS ≠ paper reproduction.
ap=ROOT/'outputs/reproducibility_audit.json'; audit=json.loads(ap.read_text()) if ap.is_file() else {}
if audit: display(pd.DataFrame([[g,audit.get(g,{}).get('status'),audit.get(g,{}).get('reason')] for g in ['G0_reduced_baseline','G1_official_checkpoint','G2_method_faithfulness','G3_training_reproduction']],columns=['gate','status','reason']))
pre=(ROOT/'outputs/ted3_reproduction/preflight/input_directories.json').is_file()
check=[['Dataset verified',all(p.is_dir() for p in D.values()) and pre],['Environment verified',True],['Training configuration available',(ROOT/'configs/paper.yaml').is_file()],['Evaluation scripts available',(ROOT/'scripts/compute_article_metrics.py').is_file()],['Figures generated',bool(figs)],['Tables generated',bool(list((ROOT/'outputs').rglob('*.csv'))) if (ROOT/'outputs').exists() else False]]
display(pd.DataFrame(check,columns=['Checklist','PASS']))
print('Reduced seed:',json.loads((ROOT/'outputs/final/run_manifest.json').read_text()).get('seed') if (ROOT/'outputs/final/run_manifest.json').is_file() else 'manifest unavailable')


## 11. Final Research Summary
- reduced path یک measured end-to-end teacher/student re-simulation است، نه numerical paper reproduction.
- `run_full.py` و `run_official_inference.py` مسیرهای واقعی full architecture/evaluation هستند ولی خودکار اجرا نمی‌شوند.
- TED3 بدون identity-equivalence evidence، exact TSI15k نامیده نمی‌شود.
- stageهای بدون evidence باید blocked/incomplete گزارش شوند.
- `vendor/SemiT-SAM` gitignored و با pinned bootstrap بازسازی می‌شود.
- full 26,250 iterations در این Notebook اجرا نشده است.

> **Measured reduced-scale re-simulation; not numerically equivalent to the published full-scale experiment.**
